# Walmart sales forecasting — XGBoost baseline

This notebook trains a leakage-safe global XGBoost baseline and tracks the experiment in Weights & Biases.

**Baseline scope**
- chronological 52-week validation split (never a random split)
- store, department, calendar, store metadata, markdown, and economic features
- Kaggle weighted MAE (holiday rows receive weight 5, other rows weight 1)
- the same weights are used during fitting and validation
- early stopping on validation weighted MAE
- W&B configuration, per-iteration metrics, summary metrics, tables, and artifacts

Sales lags are intentionally excluded from this first baseline. Validation lags built from observed validation targets would not match multi-step test inference and would produce optimistic results. A later experiment can add recursive or direct multi-horizon lag forecasting.

## 1. Colab setup

In Colab, select **Runtime → Change runtime type → T4 GPU**, clone/open the repository, and make the repository root the working directory. Your local W&B login is not available inside Colab, so `wandb.login()` below will ask for a key stored in a Colab secret or entered interactively.

In [ ]:
%pip install -q "xgboost>=3.0,<4" "wandb>=0.19,<1"

In [ ]:
from __future__ import annotations

import json
import math
import os
import platform
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import wandb
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
from wandb.integration.xgboost import WandbCallback

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", 100)
print({
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgb.__version__,
    "wandb": wandb.__version__,
})

## 2. Configuration and W&B authentication

`DEVICE="cuda"` uses the Colab GPU. Change it to `"cpu"` for a CPU runtime. Set `RUN_FINAL_REFIT=True` only after the validation run looks correct; that optional step retrains on all labeled rows and creates a Kaggle submission.

In [ ]:
CONFIG = {
    "seed": SEED,
    "validation_weeks": 52,
    "holiday_weight": 5.0,
    "objective": "reg:absoluteerror",
    "eval_metric": "mae",
    "n_estimators": 3000,
    "learning_rate": 0.03,
    "max_depth": 8,
    "min_child_weight": 5,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "early_stopping_rounds": 100,
    "tree_method": "hist",
    "device": "cuda",  # change to "cpu" when no CUDA GPU is available
    "run_final_refit": False,
    "log_dataset_artifact": True,
}

WANDB_ENTITY = "kende23-n-a"
WANDB_PROJECT = "Walmart-Recruiting---Store-Sales-Forecasting"

# In Colab this can read a WANDB_API_KEY secret; otherwise it prompts safely.
wandb.login(key=os.environ.get("WANDB_API_KEY"), relogin=False)
CONFIG

## 3. Load and validate data

In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "train.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find data/train.csv. In Colab, cd into the cloned repository first."
    )


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "artifacts" / "xgboost_baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_raw = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["Date"])
test_raw = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["Date"])
features = pd.read_csv(DATA_DIR / "features.csv", parse_dates=["Date"])
stores = pd.read_csv(DATA_DIR / "stores.csv")

required_train = {"Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"}
required_test = {"Store", "Dept", "Date", "IsHoliday"}
required_features = {"Store", "Date", "IsHoliday"}
required_stores = {"Store", "Type", "Size"}
assert required_train.issubset(train_raw.columns)
assert required_test.issubset(test_raw.columns)
assert required_features.issubset(features.columns)
assert required_stores.issubset(stores.columns)
assert not train_raw.duplicated(["Store", "Dept", "Date"]).any()
assert not test_raw.duplicated(["Store", "Dept", "Date"]).any()

display(pd.DataFrame({
    "rows": [len(train_raw), len(test_raw), len(features), len(stores)],
    "min_date": [train_raw.Date.min(), test_raw.Date.min(), features.Date.min(), pd.NaT],
    "max_date": [train_raw.Date.max(), test_raw.Date.max(), features.Date.max(), pd.NaT],
}, index=["train", "test", "features", "stores"]))

## 4. Feature engineering

The transformation uses only columns available for both train and test. XGBoost handles numeric missing values natively. Store type is converted to a stable integer mapping; no target encoding is used.

In [ ]:
TYPE_MAP = {"A": 0, "B": 1, "C": 2}


def build_features(
    sales_frame: pd.DataFrame,
    external: pd.DataFrame,
    store_metadata: pd.DataFrame,
) -> pd.DataFrame:
    frame = sales_frame.copy()
    external_no_holiday = external.drop(columns="IsHoliday")
    frame = frame.merge(
        external_no_holiday,
        on=["Store", "Date"],
        how="left",
        validate="many_to_one",
    )
    frame = frame.merge(store_metadata, on="Store", how="left", validate="many_to_one")

    iso = frame["Date"].dt.isocalendar()
    frame["Year"] = frame["Date"].dt.year.astype("int16")
    frame["Month"] = frame["Date"].dt.month.astype("int8")
    frame["WeekOfYear"] = iso.week.astype("int8")
    frame["Quarter"] = frame["Date"].dt.quarter.astype("int8")
    frame["DaysFromStart"] = (frame["Date"] - pd.Timestamp("2010-02-05")).dt.days.astype("int16")
    frame["WeekSin"] = np.sin(2 * np.pi * frame["WeekOfYear"] / 52.0)
    frame["WeekCos"] = np.cos(2 * np.pi * frame["WeekOfYear"] / 52.0)
    frame["MonthSin"] = np.sin(2 * np.pi * frame["Month"] / 12.0)
    frame["MonthCos"] = np.cos(2 * np.pi * frame["Month"] / 12.0)
    frame["TotalMarkDown"] = frame[
        ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
    ].sum(axis=1, min_count=1)
    frame["Type"] = frame["Type"].map(TYPE_MAP).astype("int8")
    frame["IsHoliday"] = frame["IsHoliday"].astype("int8")

    assert frame["Size"].notna().all(), "Store metadata is missing after merge"
    return frame.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)


train = build_features(train_raw, features, stores)
test = build_features(test_raw, features, stores)

NON_FEATURE_COLUMNS = {"Date", "Weekly_Sales"}
FEATURE_COLUMNS = [column for column in train.columns if column not in NON_FEATURE_COLUMNS]
assert FEATURE_COLUMNS == [column for column in test.columns if column not in NON_FEATURE_COLUMNS]
assert all(pd.api.types.is_numeric_dtype(train[column]) for column in FEATURE_COLUMNS)

print(f"Features: {len(FEATURE_COLUMNS)}")
display(train[FEATURE_COLUMNS].dtypes.rename("dtype").to_frame())
display(train.head())

## 5. Chronological split and metric

The final 52 weeks form validation. This includes the 2011 Thanksgiving and Christmas period while keeping every validation date strictly later than every training date.

In [ ]:
def weighted_mae(
    y_true: np.ndarray | pd.Series,
    y_pred: np.ndarray | pd.Series,
    is_holiday: np.ndarray | pd.Series,
    holiday_weight: float = 5.0,
) -> float:
    weights = np.where(np.asarray(is_holiday, dtype=bool), holiday_weight, 1.0)
    return float(np.average(np.abs(np.asarray(y_true) - np.asarray(y_pred)), weights=weights))


validation_start = train["Date"].max() - pd.Timedelta(weeks=CONFIG["validation_weeks"] - 1)
train_mask = train["Date"] < validation_start
valid_mask = ~train_mask

X_train = train.loc[train_mask, FEATURE_COLUMNS]
y_train = train.loc[train_mask, "Weekly_Sales"]
X_valid = train.loc[valid_mask, FEATURE_COLUMNS]
y_valid = train.loc[valid_mask, "Weekly_Sales"]

w_train = np.where(train.loc[train_mask, "IsHoliday"].to_numpy(dtype=bool), CONFIG["holiday_weight"], 1.0)
w_valid = np.where(train.loc[valid_mask, "IsHoliday"].to_numpy(dtype=bool), CONFIG["holiday_weight"], 1.0)

assert train.loc[train_mask, "Date"].max() < train.loc[valid_mask, "Date"].min()
assert X_train.columns.tolist() == X_valid.columns.tolist()

split_summary = {
    "train_rows": len(X_train),
    "valid_rows": len(X_valid),
    "train_start": str(train.loc[train_mask, "Date"].min().date()),
    "train_end": str(train.loc[train_mask, "Date"].max().date()),
    "valid_start": str(train.loc[valid_mask, "Date"].min().date()),
    "valid_end": str(train.loc[valid_mask, "Date"].max().date()),
    "feature_count": len(FEATURE_COLUMNS),
}
split_summary

## 6. Start the W&B run

The raw CSVs are logged as a versioned dataset artifact. W&B deduplicates unchanged artifact contents.

In [ ]:
run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    job_type="train",
    name="xgboost-static-baseline",
    tags=["baseline", "xgboost", "time-split", "wmae"],
    config={**CONFIG, **split_summary, "features": FEATURE_COLUMNS},
    save_code=True,
)

if CONFIG["log_dataset_artifact"]:
    dataset_artifact = wandb.Artifact(
        name="walmart-recruiting-raw-data",
        type="dataset",
        description="Raw Kaggle Walmart recruiting CSV files used by the baseline.",
        metadata=split_summary,
    )
    for filename in ["train.csv", "test.csv", "features.csv", "stores.csv"]:
        dataset_artifact.add_file(DATA_DIR / filename, name=filename)
    run.log_artifact(dataset_artifact, aliases=["latest"])

run

## 7. Sanity benchmark

A training-period Store/Dept median provides context for the XGBoost score. Unseen pairs fall back to department median and then the global median.

In [ ]:
history = train.loc[train_mask, ["Store", "Dept", "Weekly_Sales"]]
valid_keys = train.loc[valid_mask, ["Store", "Dept"]]
pair_median = history.groupby(["Store", "Dept"])["Weekly_Sales"].median()
dept_median = history.groupby("Dept")["Weekly_Sales"].median()
global_median = float(history["Weekly_Sales"].median())

naive_pred = pd.Series(
    [pair_median.get((store, dept), np.nan) for store, dept in valid_keys.itertuples(index=False)],
    index=valid_keys.index,
    dtype="float64",
)
naive_pred = naive_pred.fillna(valid_keys["Dept"].map(dept_median)).fillna(global_median)
naive_wmae = weighted_mae(
    y_valid,
    naive_pred,
    train.loc[valid_mask, "IsHoliday"],
    CONFIG["holiday_weight"],
)
run.log({"baseline/validation_wmae": naive_wmae})
print(f"Store/Dept median validation WMAE: {naive_wmae:,.2f}")

## 8. Train XGBoost

This is the expensive cell. Run it in Colab. The W&B callback logs training/validation MAE each boosting round and a gain-based feature-importance chart. Because validation weights are supplied, XGBoost's validation MAE is the competition WMAE.

In [ ]:
model_params = {
    key: CONFIG[key]
    for key in [
        "objective", "eval_metric", "n_estimators", "learning_rate", "max_depth",
        "min_child_weight", "subsample", "colsample_bytree", "reg_alpha", "reg_lambda",
        "early_stopping_rounds", "tree_method", "device",
    ]
}

model = xgb.XGBRegressor(
    **model_params,
    random_state=CONFIG["seed"],
    n_jobs=-1,
    callbacks=[
        WandbCallback(
            log_model=False,
            log_feature_importance=True,
            importance_type="gain",
            define_metric=True,
        )
    ],
)

model.fit(
    X_train,
    y_train,
    sample_weight=w_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    sample_weight_eval_set=[w_train, w_valid],
    verbose=50,
)

print(f"Best iteration: {model.best_iteration}")
print(f"Best XGBoost validation WMAE: {model.best_score:,.2f}")

## 9. Evaluate and log diagnostics

In [ ]:
valid_pred = model.predict(X_valid)
valid_holiday = train.loc[valid_mask, "IsHoliday"].to_numpy(dtype=bool)

metrics = {
    "validation/wmae": weighted_mae(y_valid, valid_pred, valid_holiday, CONFIG["holiday_weight"]),
    "validation/mae": float(mean_absolute_error(y_valid, valid_pred)),
    "validation/rmse": float(math.sqrt(mean_squared_error(y_valid, valid_pred))),
    "validation/holiday_mae": float(mean_absolute_error(y_valid[valid_holiday], valid_pred[valid_holiday])),
    "validation/non_holiday_mae": float(mean_absolute_error(y_valid[~valid_holiday], valid_pred[~valid_holiday])),
    "validation/improvement_over_median_pct": float(
        100 * (naive_wmae - weighted_mae(y_valid, valid_pred, valid_holiday, CONFIG["holiday_weight"])) / naive_wmae
    ),
    "model/best_iteration": int(model.best_iteration),
    "model/best_score": float(model.best_score),
}
run.log(metrics)
run.summary.update(metrics)
display(pd.Series(metrics, name="value").to_frame())

validation_results = train.loc[
    valid_mask, ["Store", "Dept", "Date", "IsHoliday", "Weekly_Sales"]
].copy()
validation_results["Prediction"] = valid_pred
validation_results["AbsoluteError"] = np.abs(
    validation_results["Weekly_Sales"] - validation_results["Prediction"]
)
validation_results["Date"] = validation_results["Date"].dt.strftime("%Y-%m-%d")

# Keep the interactive table responsive while retaining a deterministic sample.
table_rows = validation_results.nlargest(2500, "AbsoluteError")
random_rows = validation_results.sample(min(2500, len(validation_results)), random_state=SEED)
wandb_rows = pd.concat([table_rows, random_rows]).drop_duplicates().reset_index(drop=True)
run.log({"validation/predictions": wandb.Table(dataframe=wandb_rows)})

In [ ]:
importance = (
    pd.DataFrame({"feature": FEATURE_COLUMNS, "importance": model.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
display(importance.head(20))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
top = importance.head(20).sort_values("importance")
axes[0].barh(top["feature"], top["importance"])
axes[0].set_title("Top 20 XGBoost feature importances")
axes[0].set_xlabel("Importance")

sample = validation_results.sample(min(15000, len(validation_results)), random_state=SEED)
axes[1].scatter(sample["Weekly_Sales"], sample["Prediction"], alpha=0.15, s=8)
limits = [sample[["Weekly_Sales", "Prediction"]].min().min(), sample[["Weekly_Sales", "Prediction"]].max().max()]
axes[1].plot(limits, limits, "r--", linewidth=1)
axes[1].set_title("Validation: actual vs predicted")
axes[1].set_xlabel("Actual Weekly_Sales")
axes[1].set_ylabel("Predicted Weekly_Sales")
plt.tight_layout()

run.log({"validation/diagnostics": wandb.Image(fig)})
plt.show()

## 10. Save and log the validation model

JSON is XGBoost's stable model format. Feature order and run metadata are saved beside it, then all files are logged as one model artifact.

In [ ]:
model_path = OUTPUT_DIR / "xgboost_baseline_validation.json"
feature_path = OUTPUT_DIR / "feature_columns.json"
importance_path = OUTPUT_DIR / "feature_importance.csv"
metrics_path = OUTPUT_DIR / "validation_metrics.json"

model.save_model(model_path)
feature_path.write_text(json.dumps(FEATURE_COLUMNS, indent=2))
importance.to_csv(importance_path, index=False)
metrics_path.write_text(json.dumps(metrics, indent=2))

model_artifact = wandb.Artifact(
    name="xgboost-static-baseline",
    type="model",
    description="Leakage-safe XGBoost baseline trained with a chronological validation split.",
    metadata={**metrics, **split_summary},
)
model_artifact.add_dir(OUTPUT_DIR)
run.log_artifact(model_artifact, aliases=["validation", "latest"])
print(f"Saved model files to {OUTPUT_DIR}")

## 11. Optional final refit and Kaggle submission

This cell does nothing while `CONFIG["run_final_refit"]` is `False`. When enabled, it uses the best validation iteration count, refits on all labeled data, writes predictions in Kaggle's required `Store_Dept_Date,Weekly_Sales` format, and logs the final model and submission.

In [ ]:
if CONFIG["run_final_refit"]:
    final_rounds = int(model.best_iteration) + 1
    final_params = {
        key: value
        for key, value in model_params.items()
        if key not in {"n_estimators", "early_stopping_rounds"}
    }
    final_model = xgb.XGBRegressor(
        **final_params,
        n_estimators=final_rounds,
        random_state=CONFIG["seed"],
        n_jobs=-1,
    )
    all_weights = np.where(train["IsHoliday"].to_numpy(dtype=bool), CONFIG["holiday_weight"], 1.0)
    final_model.fit(train[FEATURE_COLUMNS], train["Weekly_Sales"], sample_weight=all_weights, verbose=False)

    test_pred = final_model.predict(test[FEATURE_COLUMNS])
    submission = pd.DataFrame({
        "Id": (
            test_raw["Store"].astype(str)
            + "_" + test_raw["Dept"].astype(str)
            + "_" + test_raw["Date"].dt.strftime("%Y-%m-%d")
        ),
        "Weekly_Sales": test_pred,
    })
    assert len(submission) == len(test_raw)
    assert submission["Weekly_Sales"].notna().all()

    final_model_path = OUTPUT_DIR / "xgboost_baseline_final.json"
    submission_path = OUTPUT_DIR / "submission_xgboost_baseline.csv"
    final_model.save_model(final_model_path)
    submission.to_csv(submission_path, index=False)

    final_artifact = wandb.Artifact(
        name="xgboost-static-baseline-final",
        type="model",
        metadata={"n_estimators": final_rounds, "training_rows": len(train)},
    )
    final_artifact.add_file(final_model_path)
    final_artifact.add_file(feature_path)
    final_artifact.add_file(submission_path)
    run.log_artifact(final_artifact, aliases=["production-candidate", "latest"])
    run.log({"test/prediction_distribution": wandb.Histogram(test_pred)})
    print(f"Saved submission to {submission_path}")
else:
    print("Final refit skipped. Set CONFIG['run_final_refit'] = True and rerun to create a submission.")

## 12. Finish the W&B run

Always finish the run so pending metrics and artifacts are uploaded.

In [ ]:
run.finish()

## Next experiment

Add historical sales features only with an inference strategy that matches the test horizon. Suitable options are recursive weekly prediction, one model per forecast horizon, or lag-52 features where the required historical value is available. Compare every experiment against this baseline using the same chronological split and WMAE.